# 3D monolayer lift-off — apical tension drives epithelial folding

_Investigation `monolayer-liftoff` — coder reproduction notebook._

**Question.** Can differential surface tension alone — a high line tension on the apical edges of a 3D tyssue monolayer — buckle the apical surface out of plane, reproducing the mechanical first step of lumen / dome morphogenesis, inside the workspace's process-bigraph EulerSolver?

A demonstration that apical-edge line tension alone makes a 3D tyssue monolayer fold its apical surface out of plane — the mechanical seed of a lumen — running in the same process-bigraph EulerSolver used for the 2D studies.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/vivarium-tyssue').is_dir():
    REPO = Path('/Users/eranagmon/code/vivarium-tyssue')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from vivarium_tyssue.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---
from scripts.run_study_sims import run_study
from scripts.render_study_viz import _render_one

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `lumen-liftoff`

**Question.** Does a high line tension placed on the apical-surface edges of a 3D tyssue monolayer buckle that surface out of plane — the mechanical first step of lumen / dome morphogenesis — purely from differential tension, with no growth or cell-fate events?

**Objective.** Build the 3D monolayer lift-off composite (a flat sheet extruded into a polyhedral monolayer with an apical-edge "purse-string" line tension) and demonstrate, through 3D animation, a side profile, and morphology time-series, that the apical surface deforms out of plane against cell-volume elasticity.

**Hypothesis.** With apical line tension >> bulk, the apical surface cannot stay flat: the purse-string shortens the apical perimeter while cell volume is conserved, so the surface invaginates (interior cells sink relative to the boundary rim) and the out-of-plane amplitude grows monotonically over the relaxation.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `liftoff` | `monolayer_liftoff` | 100 | interval=0.1 |

Declared parameter sets (`study.yaml` variants):

- **reference** — `interval=0.1`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `monolayer_liftoff`** — `spec_monolayer_liftoff` (a plain, editable dict)


In [ ]:
from pbg_superpowers.composite_spec import load_spec
spec_monolayer_liftoff = load_spec(REPO / 'vivarium_tyssue/composites/monolayer_liftoff.composite.yaml')
describe_spec(spec_monolayer_liftoff)

In [ ]:
# === Edit parameters for composite 'monolayer_liftoff' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_monolayer_liftoff['parameters']['interval']['default'] = 0.1

# process 'Tyssue'  (local:EulerSolver)
# spec_monolayer_liftoff['state']['Tyssue']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_monolayer_liftoff['state']['Tyssue']['config']['name'] = 'Monolayer Lift-Off'
spec_monolayer_liftoff['state']['Tyssue']['config']['eptm'] = 'workspace/datasets/monolayer_box.hf5'
spec_monolayer_liftoff['state']['Tyssue']['config']['tissue_type'] = 'Monolayer'
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['cell_df']['vol_elasticity'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['cell_df']['prefered_vol'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['cell_df']['is_alive'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['face_df']['area_elasticity'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['face_df']['prefered_area'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['face_df']['contractility'] = 0.05
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['face_df']['is_alive'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['edge_df']['line_tension']['apical'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['edge_df']['line_tension']['default'] = 0.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['edge_df']['is_active'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['vert_df']['viscosity'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['parameters']['vert_df']['is_alive'] = 1.0
spec_monolayer_liftoff['state']['Tyssue']['config']['geom'] = 'MonolayerGeometry'
spec_monolayer_liftoff['state']['Tyssue']['config']['effectors'] = ['CellVolumeElasticity', 'FaceAreaElasticity', 'FaceContractility', 'LineTension']
spec_monolayer_liftoff['state']['Tyssue']['config']['ref_effector'] = 'FaceAreaElasticity'
spec_monolayer_liftoff['state']['Tyssue']['config']['factory'] = 'model_factory'
spec_monolayer_liftoff['state']['Tyssue']['config']['settings']['threshold_length'] = 0.03
spec_monolayer_liftoff['state']['Tyssue']['config']['auto_reconnect'] = False

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: lumen-liftoff ===
STUDY = 'lumen-liftoff'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_liftoff = 100
INTERVAL_liftoff = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # sim 'liftoff' <- edited composite spec spec_monolayer_liftoff ('monolayer_liftoff')
        run_study(STUDY, 'liftoff', spec_monolayer_liftoff, STEPS_liftoff, INTERVAL_liftoff)
    print(f'ran 1 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Lift-off side profile**


In [ ]:
# Lift-off side profile
show_viz(_render_one('local:TissueSheetGif', {'title': 'Lift-off side profile (x-z)', 'coords': ['x', 'z'], 'num_frames': 40, 'sources': ['liftoff'], 'caption': 'The run projected onto the x-z plane (the monolayer viewed edge-on), so the vertical motion is read directly: the apical surface (top, starting at z=0) deforms downward and inward into a curved profile while the basal surface (bottom, z=-1) barely moves — apical tension alone folds the monolayer. This side view is the clearest read on the 3D shape change; a true rotating 3D animation needs ipyvolume / a faster 3D renderer (see discovery_implications).'}, RUNS_DB, STUDY_YAML))

**Apical deformation over time**


In [ ]:
# Apical deformation over time
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Apical surface deformation over time', 'observables': ['apical_amplitude', 'apical_mean_z', 'apical_min_z'], 'caption': 'apical_amplitude (max-min z of the apical surface) grows monotonically from 0 to ~2.3 — the surface is increasingly non-planar. apical_mean_z drifts below 0 and apical_min_z falls further, i.e. the surface sinks and curves rather than rising uniformly.'}, RUNS_DB, STUDY_YAML))

**Dome / invagination over time**


In [ ]:
# Dome / invagination over time
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Interior vs boundary cell height (dome / invagination)', 'observables': ['dome_height', 'interior_cell_z', 'boundary_cell_z'], 'caption': 'dome_height (interior cell z minus boundary rim z) goes increasingly negative (~-1.5), meaning interior cells sink below the boundary rim — the apical surface invaginates inward, the geometry that would seed a lumen/pit.'}, RUNS_DB, STUDY_YAML))

**Lift-off profile snapshots**


In [ ]:
# Lift-off profile snapshots
show_viz(_render_one('local:TissueSheetSnapshots', {'title': 'Lift-off profile snapshots (x-z)', 'coords': ['x', 'z'], 'n_panels': 6, 'caption': 'Six x-z stills across the run; the flat apical line at t0 progressively curves out of plane as the apical purse-string contracts.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| apical_liftoff_occurs | kind=last path=apical_amplitude | op gt threshold 1.0 |
| interior_invaginates | kind=last path=dome_height | op lt threshold -0.5 |


## Open decisions
- Whether to add a flat control (apical tension 0) and a tension sweep to turn this demonstration into a controlled, parameterized study.
- Whether to enable topological rearrangements (T1) to progress the invagination toward a closed lumen.
